In [2]:
import pandas as pd
import tkinter as tk
from tkinter import filedialog, messagebox, simpledialog, ttk

import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.worksheet.hyperlink import Hyperlink
from openpyxl.cell.cell import MergedCell

import math
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


class BrakeVCApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Brake VC Auto Processor + Summary Generator")
        self.root.geometry("600x500")

        self.file_path = ""
        self.autofill_path = ""
        self.autofill_df = None

        self.workbook = openpyxl.Workbook()
        try:
            self.workbook.remove(self.workbook.active)
        except:
            pass

        self.create_widgets()

    # ---------------------------------------------------------
    def create_widgets(self):
        style = ttk.Style()
        style.configure("TButton", font=("Arial", 12), padding=5)
        style.configure("TLabel", font=("Arial", 10))

        ttk.Button(self.root, text="Select Brake VC CSV", command=self.browse_csv).grid(row=0, column=0, padx=10, pady=10)
        self.csv_label = ttk.Label(self.root, text="No CSV selected")
        self.csv_label.grid(row=0, column=1, sticky="w")

        ttk.Button(self.root, text="Select Auto-Fill Excel", command=self.browse_autofill).grid(row=1, column=0, padx=10, pady=10)
        self.autofill_label = ttk.Label(self.root, text="No Excel selected")
        self.autofill_label.grid(row=1, column=1, sticky="w")

        ttk.Button(self.root, text="Process All Rows", command=self.process_all).grid(row=2, column=0, columnspan=2, pady=10)
        ttk.Button(self.root, text="Create Summary Sheet", command=self.add_summary_sheet).grid(row=3, column=0, columnspan=2, pady=10)
        ttk.Button(self.root, text="Save Workbook", command=self.save_workbook).grid(row=4, column=0, columnspan=2, pady=10)

    # ---------------------------------------------------------
    def browse_csv(self):
        self.file_path = filedialog.askopenfilename(filetypes=[("CSV files", "*.csv")])
        if self.file_path:
            self.csv_label.config(text=self.file_path.split("/")[-1])

    def browse_autofill(self):
        self.autofill_path = filedialog.askopenfilename(filetypes=[("Excel files", "*.xlsx;*.xls")])
        if not self.autofill_path:
            return
        self.autofill_label.config(text=self.autofill_path.split("/")[-1])

        try:
            self.autofill_df = pd.read_excel(self.autofill_path)
        except Exception as e:
            messagebox.showerror("Error Reading Autofill Excel", str(e))
            return

        required = ['Start time', 'End time', 'station', 'dir', 'Loco type', 'Test type', 'signal location']
        if not all(col in self.autofill_df.columns for col in required):
            messagebox.showerror("Error", f"AutoFill Excel must contain:\n{required}")
            self.autofill_df = None

    # ---------------------------------------------------------
    def convert_timestamp(self, timestamp, ncol):
        try:
            if ncol == 58:
                try:
                    return datetime.strptime(timestamp, "%H:%M:%S.%f %d/%m/%Y").strftime("%H:%M:%S")
                except:
                    return datetime.strptime(timestamp, "%H:%M:%S:%f %d/%m/%Y").strftime("%H:%M:%S")
            else:
                return datetime.strptime(timestamp, "%H:%M:%S %d/%m/%Y").strftime("%H:%M:%S")
        except:
            return timestamp

    # ---------------------------------------------------------
    def extract_data(self, start, end, loco_type, test_type):
        hdr4 = [
            "Spd","SBp","BpR","LePr","Mr","Eb","LeBL","LeMd","MdEb","ClngSpd","MaV","MA","ToV","ToSpd","ToDist",
            "BrkDly","AcDc","GrDc","FlDc","DsPrmSpd","DsTrSpd","MdSpdLm","AbsLoc","ClStBp","EmStBp","ToStBp",
            "MaStBp","MdStBp","UnStBp","SoStBp","SspStBp","ClLeLv","EmLeLv","MaLeLv","ToLeLv","MdLeLv",
            "UnLeLv","SoLeLv","SspLeLv","EbCnFlags","ClPrSp","EmPrSp","ToPrSp","MaPrSp","SspPrSp","ClTrSp",
            "EmTrSp","ToTrSp","MaTrSp","SspTrSp","FltType","FltCode","TsrSpdVld","TsrCrSpd","TsrPrvSpd",
            "TsrAprSpd","TsrAprSpdVld","TimeStamp"
        ]

        hdr3 = [
            "Spd","SBp","BpR","LePr","Mr","Eb","LeBL","LeMd","MdEb","ClngSpd","MaV","MA","ToV","ToSpd","ToDist",
            "BrkDly","AcDc","GrDc","FlDc","DsPrmSpd","DsTrSpd","MdSpdLm","AbsLoc","ClStBp","EmStBp","ToStBp",
            "MaStBp","MdStBp","UnStBp","SoStBp","SspStBp","ClLeLv","EmLeLv","MaLeLv","ToLeLv","MdLeLv","UnLeLv",
            "SoLeLv","SspLeLv","EbCnFlags","ClPrSp","EmPrSp","ToPrSp","MaPrSp","SspPrSp","ClTrSp","EmTrSp",
            "ToTrSp","MaTrSp","SspTrSp","FltType","FltCode","TimeStamp"
        ]

        try:
            df = pd.read_csv(self.file_path)
            if df.columns.str.contains(r'\d').any():
                raise Exception("No headers detected")
        except:
            df = pd.read_csv(self.file_path, header=None)
            if df.shape[1] == 58:
                df.columns = hdr4
            elif df.shape[1] == 53:
                df.columns = hdr3
            else:
                raise Exception("Invalid CSV Column Count")

        df["TimeStamp"] = df["TimeStamp"].apply(lambda x: self.convert_timestamp(x, df.shape[1]))

        df = df[(df["TimeStamp"] >= start) & (df["TimeStamp"] <= end)]

        if loco_type.upper() == "LE":
            cols = ['Spd','LeBL','TimeStamp','AcDc','FlDc','AbsLoc','MA','ToDist','ToSpd','BrkDly','GrDc']
        else:
            cols = ['Spd','SBp','TimeStamp','AcDc','FlDc','AbsLoc','MA','ToDist','ToSpd','BrkDly','GrDc']

        df = df[cols].copy()

        if test_type.upper() == "SPAD":
            for c in ["ToDist","ToSpd"]:
                if c in df.columns:
                    df = df.drop(columns=[c])

        start_rows = df[df['TimeStamp'] == start]

        if loco_type.upper() == "LE":
            tmp = start_rows[start_rows['LeBL'] == 4]
        else:
            tmp = start_rows[start_rows['SBp'] == 3.5] if "SBp" in df.columns else pd.DataFrame()

        if not tmp.empty:
            start_row = tmp.iloc[0]
        else:
            try:
                start_row = start_rows.iloc[0]
            except:
                start_row = df.iloc[0]

        if test_type.upper() == "SPAD":
            end_rows = df[df['TimeStamp'] == end]
            tmp = end_rows[end_rows['Spd'] == 0]
            if not tmp.empty:
                end_row = tmp.iloc[0]
            else:
                try:
                    end_row = end_rows.iloc[-1]
                except:
                    end_row = df.iloc[-1]
        else:
            try:
                end_row = df[df['TimeStamp'] == end].iloc[-1]
            except:
                end_row = df.iloc[-1]

        mid = df[(df['TimeStamp'] > start) & (df['TimeStamp'] < end)].copy()

        if loco_type.upper() == "LE" and "LeBL" in mid.columns:
            mid = mid.drop_duplicates(subset=['TimeStamp','LeBL'])
        elif "SBp" in mid.columns:
            mid = mid.drop_duplicates(subset=['TimeStamp','SBp'])

        final = pd.concat([start_row.to_frame().T, mid, end_row.to_frame().T], ignore_index=True)

        final['DC of loco'] = final['Spd'].diff().abs() * (5/18)
        final['DC of loco'] = final['DC of loco'].fillna(0).round(2)
        final['Distance moved'] = final['AbsLoc'].diff().abs().fillna(0).cumsum()
        final['Speed'] = final['Spd']

        return final

    # ---------------------------------------------------------
    def process_all(self):
        if not self.file_path:
            messagebox.showerror("Error", "No CSV Selected")
            return
        if self.autofill_df is None:
            messagebox.showerror("Error", "No AutoFill Excel Selected")
            return

        for _, row in self.autofill_df.iterrows():
            start = str(row['Start time']).replace(" ", ":")
            end = str(row['End time']).replace(" ", ":")
            station = str(row['station']).strip()
            direction = str(row['dir']).strip()
            loco = str(row['Loco type']).strip()
            test = str(row['Test type']).strip()

            sheet_name = f"{station}_{direction}"
            if test.lower() == "loop line test":
                sheet_name += "_LL"

            df_out = self.extract_data(start, end, loco, test)

            base = sheet_name
            c = 1
            while sheet_name in self.workbook.sheetnames:
                sheet_name = f"{base}_{c}"
                c += 1

            ws = self.workbook.create_sheet(sheet_name)

            for r in dataframe_to_rows(df_out, index=False, header=True):
                ws.append(r)

            header_fill = PatternFill(start_color="BDD7EE", fill_type="solid")
            thin = Border(left=Side(style="thin"), right=Side(style="thin"),
                          top=Side(style="thin"), bottom=Side(style="thin"))

            for cell in ws[1]:
                cell.fill = header_fill
                cell.font = Font(bold=True)
                cell.alignment = Alignment(horizontal="center")
                cell.border = thin

            ws.freeze_panes = "A2"

            for col_cells in ws.columns:
                maxlen = 0
                for c in col_cells:
                    if isinstance(c, MergedCell):
                        continue
                    try:
                        val = c.value
                        l = len(str(val)) if val else 0
                        maxlen = max(maxlen, l)
                    except:
                        pass

                real = next((x for x in col_cells if not isinstance(x, MergedCell)), None)
                if real:
                    ws.column_dimensions[real.column_letter].width = maxlen + 2

                for c in col_cells:
                    try:
                        c.border = thin
                    except:
                        pass

            messagebox.showinfo("Sheet Added", f"{sheet_name} added successfully!")

        messagebox.showinfo("Done", "All sheets processed")

    # ---------------------------------------------------------
    def add_summary_sheet(self):
        if "Summary" in self.workbook.sheetnames:
            del self.workbook["Summary"]

        summary = self.workbook.create_sheet("Summary")

        date = simpledialog.askstring("Input", "Enter date (DD-MM-YYYY):")
        loco_id = simpledialog.askstring("Input", "Loco ID:")
        formation = simpledialog.askstring("Input", "Formation (LE / Train with coaches):")

        if not date or not loco_id or not formation:
            messagebox.showerror("Error", "All fields are required")
            return

        # ---------------------
        # SPAD TABLE
        # ---------------------
        headers_1 = [
            '#', 'Date', 'LE / Formation', 'Station', 'Gradient', 'Direction',
            'Braking delay, sec', 'DC, m/sec/sec', 'Initial speed, kmph',
            'Stopping distance to EOA, mtr', 'Signal Location', 'Loco stopped location',
            'Brake applied location', 'Distance to signal when brake applied (mtrs)',
            'Braking distance (mtrs)'
        ]

        title1 = summary.cell(row=1, column=1,
                              value=f"DC and brake delay time configuration for SPAD Test in {loco_id} ({formation})")
        title1.font = Font(bold=True, size=13)
        title1.fill = PatternFill(start_color="ffc493", fill_type="solid")
        title1.alignment = Alignment(horizontal="center", vertical="center")
        summary.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(headers_1))

        # SPAD headers (row 2)
        for col, h in enumerate(headers_1, start=1):
            c = summary.cell(row=2, column=col, value=h)
            c.font = Font(bold=True)
            c.fill = PatternFill("solid", fgColor="e0e4f4")
            c.border = Border(left=Side(style='thin'), right=Side(style='thin'),
                              top=Side(style='thin'), bottom=Side(style='thin'))
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        summary.row_dimensions[2].height = 35

        row_num = 3
        serial = 1

        for sh in self.workbook.sheetnames:
            if sh == "Summary" or sh.endswith("_LL"):
                continue

            station, direction = sh.rsplit("_", 1)

            match = self.autofill_df[
                (self.autofill_df['station'].astype(str).str.strip() == station) &
                (self.autofill_df['dir'].astype(str).str.strip() == direction)
            ]

            if match.empty:
                continue

            signal_loc = float(match['signal location'].iloc[0])
            ws = self.workbook[sh]

            headers = [c.value for c in ws[1]]

            try:
                idx_gr = headers.index('GrDc') + 1
                idx_bd = headers.index('BrkDly') + 1
                idx_dc = headers.index('AcDc') + 1
                idx_spd = headers.index('Spd') + 1
                idx_abs = headers.index('AbsLoc') + 1
            except:
                continue

            gradient = ws.cell(row=2, column=idx_gr).value
            braking_delay = ws.cell(row=2, column=idx_bd).value
            dc = ws.cell(row=2, column=idx_dc).value
            initial_speed = ws.cell(row=2, column=idx_spd).value

            loco_stop = ws.cell(row=ws.max_row, column=idx_abs).value
            brake_applied = ws.cell(row=2, column=idx_abs).value

            stopping_dist = abs(signal_loc - loco_stop)
            dist_when_applied = abs(brake_applied - signal_loc)
            brake_dist = abs(brake_applied - loco_stop)

            row_data = [
                serial, date, loco_id, sh, gradient, direction,
                braking_delay, dc, initial_speed, stopping_dist,
                signal_loc, loco_stop, brake_applied, dist_when_applied, brake_dist
            ]

            for col, v in enumerate(row_data, start=1):
                c = summary.cell(row=row_num, column=col, value=v)
                c.alignment = Alignment(wrap_text=True)
                c.border = Border(left=Side(style='thin'), right=Side(style='thin'),
                                  top=Side(style='thin'), bottom=Side(style='thin'))

                if col == 4:  # Station column hyperlink
                    c.value = sh
                    c.hyperlink = Hyperlink(ref=c.coordinate, location=f"{sh}!A1", tooltip=f"Go to {sh}")
                    c.style = "Hyperlink"

            row_num += 1
            serial += 1

        # ---------------------
        # LOOP LINE TABLE
        # ---------------------
        headers_2 = [
            '#', 'Date', 'LE / Formation', 'Station', 'Gradient', 'Direction',
            'Braking delay, sec', 'DC, m/sec/sec', 'Initial speed, kmph',
            'Brake applied start location', 'Brake release location',
            'ABS Location at which Speed controlled to Turnout Speed',
            'ABS Location of facing Turnout', 'Distance to Loop when brake applied (mtrs)',
            'Braking distance (mtrs)', 'Loco speed before entering Turnout (kmph)',
            'Distance to Turnout when Loco attained Turnout speed (mtrs)', 'Remarks'
        ]

        title2_row = row_num + 2
        title2 = summary.cell(row=title2_row, column=1,
                              value=f"DC and brake delay time configuration for Loop line speed control with {loco_id} ({formation})")
        title2.font = Font(bold=True, size=13)
        title2.fill = PatternFill(start_color="ffc493", fill_type="solid")
        title2.alignment = Alignment(horizontal="center", vertical="center")
        summary.merge_cells(start_row=title2_row, start_column=1, end_row=title2_row, end_column=len(headers_2))

        # Loopline headers
        for col, h in enumerate(headers_2, start=1):
            c = summary.cell(row=title2_row + 1, column=col, value=h)
            c.font = Font(bold=True)
            c.fill = PatternFill("solid", fgColor="e0e4f4")
            c.border = Border(left=Side(style='thin'), right=Side(style='thin'),
                              top=Side(style='thin'), bottom=Side(style='thin'))
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        summary.row_dimensions[title2_row + 1].height = 48

        rnum = title2_row + 2
        serial2 = 1

        for sh in self.workbook.sheetnames:
            if not sh.endswith("_LL"):
                continue

            station_name = sh.replace("_LL", "")
            station, direction = station_name.rsplit("_", 1)


            match = self.autofill_df[
                (self.autofill_df['station'].astype(str).str.strip() == station) &
                (self.autofill_df['dir'].astype(str).str.strip() == direction)
            ]
            if match.empty:
                continue

            loop_entry = float(match['signal location'].iloc[0])
            ws = self.workbook[sh]
            headers = [c.value for c in ws[1]]

            try:
                idx_gr = headers.index('GrDc') + 1
                idx_bd = headers.index('BrkDly') + 1
                idx_dc = headers.index('AcDc') + 1
                idx_spd = headers.index('Spd') + 1
                idx_abs = headers.index('AbsLoc') + 1
            except:
                continue

            gradient = ws.cell(row=2, column=idx_gr).value
            braking_delay = ws.cell(row=2, column=idx_bd).value
            dc = ws.cell(row=2, column=idx_dc).value
            initial_speed = ws.cell(row=2, column=idx_spd).value

            brake_applied_start = ws.cell(row=2, column=idx_abs).value
            brake_release = None
            abs_turnout_speed = None

            idx_sbp = headers.index('SBp') + 1 if 'SBp' in headers else None
            idx_lebl = headers.index('LeBL') + 1 if 'LeBL' in headers else None

            for r in ws.iter_rows(min_row=2, max_row=ws.max_row):
                try:
                    if idx_sbp and r[idx_sbp-1].value == 5.5:
                        brake_release = r[idx_abs-1].value
                        break
                    if idx_lebl and r[idx_lebl-1].value == 0:
                        brake_release = r[idx_abs-1].value
                        break
                except:
                    continue

            for r in ws.iter_rows(min_row=2, max_row=ws.max_row):
                try:
                    if r[idx_spd-1].value is not None and r[idx_spd-1].value <= 30:
                        abs_turnout_speed = r[idx_abs-1].value
                        break
                except:
                    continue

            nearest = float('inf')
            speed_before_turnout = None

            for r in ws.iter_rows(min_row=2, max_row=ws.max_row):
                try:
                    loc = r[idx_abs-1].value
                    diff = abs(loc - loop_entry)
                    if diff < nearest:
                        nearest = diff
                        speed_before_turnout = r[idx_spd-1].value
                except:
                    continue

            dist_loop_applied = abs(brake_applied_start - loop_entry)
            braking_dist = abs(brake_applied_start - brake_release) if brake_release else None
            dist_to_turnout = abs(abs_turnout_speed - loop_entry) if abs_turnout_speed else None

            row_data = [
                serial2, date, loco_id, sh, gradient, direction, braking_delay, dc,
                initial_speed, brake_applied_start, brake_release, abs_turnout_speed,
                loop_entry, dist_loop_applied, braking_dist, speed_before_turnout,
                dist_to_turnout, ''
            ]

            for col, v in enumerate(row_data, start=1):
                c = summary.cell(row=rnum, column=col, value=v)
                c.border = Border(left=Side(style='thin'), right=Side(style='thin'),
                                  top=Side(style='thin'), bottom=Side(style='thin'))
                c.alignment = Alignment(wrap_text=True)

                if col == 4:
                    c.value = sh
                    c.hyperlink = Hyperlink(ref=c.coordinate, location=f"{sh}!A1", tooltip=f"Go to {sh}")
                    c.style = "Hyperlink"

            rnum += 1
            serial2 += 1

        # ---------------- BACK — AUTOFIT ------------------
        for col_cells in summary.columns:
            maxlen = 0
            for c in col_cells:
                if isinstance(c, MergedCell):
                    continue
                try:
                    val = c.value
                    length = len(str(val)) if val else 0
                    maxlen = max(maxlen, length)
                except:
                    pass

            real = next((x for x in col_cells if not isinstance(x, MergedCell)), None)
            if real:
                summary.column_dimensions[real.column_letter].width = maxlen + 2

        messagebox.showinfo("Success", "Summary sheet created successfully!")

    # ---------------------------------------------------------
    def save_workbook(self):
        save_path = filedialog.asksaveasfilename(defaultextension=".xlsx",
                                                 filetypes=[("Excel files", "*.xlsx")])
        if save_path:
            try:
                self.workbook.save(save_path)
                messagebox.showinfo("Saved", f"Workbook saved:\n{save_path}")
            except Exception as e:
                messagebox.showerror("Error", str(e))


# ---------------------------------------------------------
if __name__ == "__main__":
    root = tk.Tk()
    app = BrakeVCApp(root)
    root.mainloop()
